# Module 3 Challenges

### Paul Resnick's publication citation history: UMSI coauthors (continued)

This notebook is a continuation of the work done in the Module 3 lab notebook. It is highly recommended that you complete the lab before this assignment as the lab will help familiarize you with the data. We will also import functions that reflect the work you should have completed in the lab notebook.

We are again focusing on the citation history of publications authored or coauthored by Professor Paul Resnick. You are tasked with identifying publications that feature UMSI faculty members as coauthors. When you encounter an error consider applying the OILER framework to help you diagnose and resolve the issue.

__Note__: This notebook features two stacked bar charts. The [Vega-Altair](https://altair-viz.github.io/) library is used to visualize the data. You are not expected to learn Vega-Altair or implement the chart code located in the `alt_chart_bar.py` module. That said, you will be asked to write code that _produces_ the data to be passed to an `alt.Data()` object before being passed on to the `alt.Chart()` object.

The Vega-Altair data model favors tabular data in the guise of a [pandas](https://pandas.pydata.org/) `DataFrame` but generating charts from JSON objects and Python dictionaries are also supported. Since the teaching team hasn't covered Pandas yet, the notebook makes use of the Altair option to provide data in the form of a nested list of dictionaries.

In [1]:
import module_3_alt_chart_bar as bar
import csv
import json

# Constants
RESNICK_PAUL = "Resnick, Paul"

## 1.0 Retrieve the data

The JSON file `data-resnick_citations-v1p3.json` is the same data used in this week's lab. It contains an array of JSON objects, each of which
represents a publication authored or coauthored by Paul. The data was sourced from the
[Web of Science](https://clarivate.com/academia-government/scientific-and-academic-research/research-discovery-and-referencing/web-of-science/).
Originally, each publication comprised `50` name-value pairs. However, each publication has been
"thinned" by dropping name-value pairs not required for the analysis. The following publication attributes are retained:

* Publication title
* Authors
* Source title
* Publication year
* Total number of citations
* Annual average number of citations since publication
* Citation counts per year, 1992-2023 (represented as a list of nested dictionaries)

The code below reads in a JSON file as a list of dictionaries. Each dictionary represents a single publication. The process of reading a file into a hierarchically structured object is called _deserialization_.

In [2]:
filepath = "./data/data-resnick_citations-v1p3.json"
with open(filepath, "r", encoding="utf-8") as file_obj:
    publications = json.load(file_obj)

print(f"\nPublications (n={len(publications)})")


Publications (n=55)


### 1.1 Variable names

Variable names in this notebook leverage the following abbreviations. The naming
strategy is to strike a balance between brevity and readability:

* `chrt`: chart
* `cits`: citations
* `coauth`: coauthors
* `pct`: percentage
* `pub`: publication
* `pubs`: publications
* `umsi`: University of Michigan School of Information

### 1.2 Reuse lab code to get years and total citations
One of the many advantages of coding in Python is that we can create modular, reusable code for convenience and consistency. Examine the accompanying `module_3_lab_functions.py` file where we have provided a version of some of the code generated in this week's lab session so we don't have to repeat what we have already done. Execute the cell below to import the module and call functions to retreive the publication years and total citations.

In [3]:
'''
Note: it is best practice to place all import statements at the top of your .ipynb or .py file.
We are placing it here simply to demonstrate how it is being used.
'''
import module_3_lab_functions as lab

years = lab.get_years(publications)
total_cits = lab.get_total_cits(publications)

print(years)
print(total_cits)

[1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
5967


## 2.0 UMSI faculty coauthors? [1 pt]

The University of Michigan School of Information (UMSI) is a hive of research initiatives that
often involve collaboration between members of the UMSI faculty. Has Paul engaged with his UMSI
colleagues on research that resulted in publications cited by others in the wider academic
community? If so, to what extent?

Read in a JSON file of UMSI faculty members named `data-umsi-faculty.csv` that is found in the
`data` directory. You will use the list to match publication coauthors to _current_ UMSI faculty
members. Assign the return value to a variable named `faculty_data`.

__Note__: The faculty list is limited to current faculty (circa `2024`) only. It's quite possible
that Paul may have coauthored publications with colleagues who have since left the School of
Information. Compiling those names is out of scope for this exercise.

In [4]:
filepath = "./data/data-umsi-faculty.csv"
with open(filepath, "r", encoding="utf-8", newline="") as file_obj:
    faculty_data = [row for row in csv.reader(file_obj, delimiter=",")]

Take a look at the raw CSV file contents in the `data` directory. You'll see that there is a header
row with two column names, "Last Name" and "First Name". The subsequent rows contain the names of
UMSI faculty members.

Assign the headers row from `faculty_data` to the variable named `faculty_headers`. Assign the rest
of the rows to the variable named `faculty`."

In [5]:
faculty_headers = faculty_data[0]   # ['Last Name', 'First Name']
faculty = faculty_data[1:]          # remaining rows: [['Adar', 'Eytan'], ...]

In [6]:
# Hidden tests are within this cell

## 3.0 Implement `has_faculty_coauthor()` [1 pt]

To simplify evaluating each publication's "Authors" string in search of UMSI coauthors
_other than Paul_, implement the function named `has_faculty_coauthor()`. The function is
provisioned with three (`3`) parameters:

* `publication` (`dict`): A dictionary representing a publication.
* `author` (`str`): A string representing a publication author to exclude from the search.
* `faculty` (`list`): A list of UMSI faculty members.

The function returns `True` if a coauthor _other_ than the `author` (e.g., "Resnick, Paul") is a
member of the `faculty` list; otherwise the function returns `False`. Only one UMSI faculty member
other than Paul needs to be located as a publication coauthor for the function to return `True`.

__Note__: A person's name can be represented in different ways in author lists. For example:
"Erin Krupka", "Erin L Krupka", "EL Krupka", and "Krupka, Erin". You **do not need to worry about**
possible name variations in this exercise. The format is "Last Name, First Name".

In [7]:
def has_faculty_coauthor(publication, author, faculty):
    """
    Returns True if the publication's 'Authors' string contains at least one
    UMSI faculty member other than < author >.

    Parameters:
        publication (dict): A dictionary representing a single publication.
        author (str): Author to exclude from the search (e.g. "Resnick, Paul").
        faculty (list): List of [last_name, first_name] rows from the faculty CSV.

    Returns:
        bool: True if a faculty coauthor (other than author) is found, else False.
    """
    # Split into individual author strings and strip whitespace
    pub_authors = [a.strip() for a in publication["Authors"].split(";")]
    for coauthor in pub_authors:
        # Skip the excluded author
        if coauthor.lower() == author.lower():
            continue
        # Check if this coauthor is a UMSI faculty member (exact match)
        for row in faculty:
            faculty_name = f"{row[0]}, {row[1]}"
            if coauthor.lower() == faculty_name.lower():
                return True
    return False

In [8]:
# Hidden tests are within this cell

## 4.0 Group the publications [1 pt]

Now use the function `has_faculty_coauthor()` to add an additional key-value pair to each nested
publication dictionary in `publications`. The new key is `"Group"` and the associated value is either
`"UMSI coauthors"` or `"No UMSI coauthors"`.

Loop over the `publications` list. Inside the loop block implement a conditional statement
that evaluates the return value of `has_faculty_coauthor()`. Pass the arguments the function
requires to discover UMSI coauthors other than Paul. Evaluate the return value of the function call
and add the appropriate key-value pair to the publication dictionary.

In [9]:
for pub in publications:
    if has_faculty_coauthor(pub, RESNICK_PAUL, faculty):
        pub["Group"] = "UMSI coauthors"
    else:
        pub["Group"] = "No UMSI coauthors"

In [10]:
# Hidden tests are within this cell

## 5.0 Implement `get_group()` [1 pt]

Next, implement the function `get_group()`. The function is provisioned with two (`2`) parameters:

* `publications` (`list`): A list of nested dictionaries, each of which represents a publication.
* `name` (`str`): A string representing the name of the group.

The function _must_ return a new list of publications filtered on the group name (e.g., "UMSI").

In [11]:
def get_group(publications, name):
    """
    Filters publications by the value of the 'Group' key.

    Parameters:
        publications (list): List of publication dictionaries.
        name (str): Group name to filter on (e.g. "UMSI coauthors").

    Returns:
        list: Publications whose 'Group' matches name.
    """
    return [pub for pub in publications if pub["Group"] == name]

In [12]:
# Hidden tests are within this cell

## 6.0 Create the chart data

### 6.1 All publications [1 pt]

Now it's time to generate the chart data. First, call the function `lab.create_chart_data()` from the lab module and pass it
the arguments required to return a chart-friendly list of nested dictionaries representing
__all Paul's authored/coauthored publications__. Assign the return value to a variable named
`chrt_all_cits`. The `chrt_all_cits` list will be used to generate a three-year (`3`) rolling mean
of annual citation counts and provide data for a chart text object.

In [13]:
chrt_all_cits = lab.create_chart_data(publications, years)

In [14]:
# Hidden tests are within this cell

### 6.2 UMSI coauthored publications [1 pt]

Call `get_group()` and pass it the arguments required to return a list of nested dictionaries
representing the __UMSI co-authored publications__. Assign the return value to a variable named
`umsi_coauth_pubs`.

Next, call `lab.create_chart_data()` and pass it the arguments required to return a chart-friendly list
of nested dictionaries representing the UMSI co-authored publications. Assign the return value to a
variable named `chrt_umsi_coauth_cits`.

__Note:__ You only need to call the functions you've implemented. You don't need to write code to
reimplement any of the complex logic embedded in those functions.

In [15]:
umsi_coauth_pubs = get_group(publications, "UMSI coauthors")
chrt_umsi_coauth_cits = lab.create_chart_data(umsi_coauth_pubs, years, "UMSI coauthors")

In [16]:
# Hidden tests are within this cell

### 6.3 Non-UMSI coauthored publications [1 pt]

Repeat the process above for __non-UMSI coauthored publications__. Assign the return values of the
function calls to the variables `non_umsi_coauth_pubs` and `chrt_non_umsi_coauth_cits`.

In [17]:
non_umsi_coauth_pubs = get_group(publications, "No UMSI coauthors")
chrt_non_umsi_coauth_cits = lab.create_chart_data(non_umsi_coauth_pubs, years, "No UMSI coauthors")

In [18]:
# Hidden tests are within this cell

## 7.0 Generate a stacked bar chart

Now you've reached the fun part: generating a nice visualization of the data. This notebook relies
on functions located in the `alt_chart_bar.py` module (aliased as `bar`) to produce the stacked bar
chart.

In the code cell below, the data along with a title, subtitle, and custom colors and labels, are
passed to the function named `create_stacked_bar_chart()`. The chart object returned by the function
call is assigned to the variable `chart` and then displayed.

You do not have to write any additional code to generate the chart. Just run the cell below. If the
chart is not displayed or doesn't look right, apply the OILER framework to diagnose and resolve the
issue.

__Note:__ If you want to explore the Altair code, open the `alt_chart_bar.py` file. You can also try
changing the arguments that are passed to the `create_stacked_bar_chart()` function. The teaching
team recommends that you make a copy of this cell and experiment with the copy rather than modify
the original code.

In [19]:
# Generate stacked bar chart
grouped_cits = chrt_umsi_coauth_cits + chrt_non_umsi_coauth_cits
title = "Citation counts: Paul Resnick authored/coauthored publications, 1992-2023"
subtitle = (
    f"publications: n={len(publications)} | citations: n={total_cits} | three-year rolling mean"
)
custom_colors = ["#FFCB05", "#00274C"]  # maize, blue
custom_labels = {
    "UMSI coauthors": f"UMSI coauthors (n={len(umsi_coauth_pubs)})",
    "No UMSI coauthors": f"No UMSI coauthors (n={len(non_umsi_coauth_pubs)})",
}

chart = bar.create_stacked_bar_chart(
    data_grouped=grouped_cits,
    data_ungrouped=chrt_all_cits,
    x_shorthand="year:O",
    y_shorthand="citations:Q",
    grp_shorthand="group:N",
    custom_colors=custom_colors,
    custom_labels=custom_labels,
    title=title,
    subtitle=subtitle,
    rm_shorthand="rolling_mean:Q",
    width=725,
)
chart.display()

alt.LayerChart(...)

## 8.0 UMSI research collaboration

Between `2007` and `2023` Paul authored/coauthored `35` publications, `17` (`48.57%`) of which
have featured one or more current UMSI faculty members as coauthors. The UMSI coauthored publications
have netted `663` citations (`53.17%`) out of a total of `1247` citations generated during the period.

These numbers are derived from computations that you need to produce below. You will also produce
the data required to generate a second stacked bar chart that focuses on the period.


### 8.1 Publications, 2007-2023 [1 pt]

Create the following lists from the `publications` list.

1. All publications between `2007` and `2023` (inclusive).  Assign the new list to a variable named
   `pubs_2007_2023`.

2. UMSI coauthored publications between `2007` and `2023` (inclusive).  Assign the new list to a variable named `umsi_coauth_pubs_2007_2023`.

3. Non-UMSI coauthored publications between `2007` and `2023` (inclusive).  Assign the new list to a variable named `non_umsi_coauth_pubs_2007_2023`.

In [20]:
pubs_2007_2023 = [
    pub for pub in publications
    if 2007 <= pub["Publication Year"] <= 2023
]

umsi_coauth_pubs_2007_2023 = [
    pub for pub in umsi_coauth_pubs
    if 2007 <= pub["Publication Year"] <= 2023
]

non_umsi_coauth_pubs_2007_2023 = [
    pub for pub in non_umsi_coauth_pubs
    if 2007 <= pub["Publication Year"] <= 2023
]

In [21]:
# Hidden tests are within this cell

### 8.2 Publication group percentages [1 pt]

Compute the percentage values of UMSI coauthored publications and non-UMSI coauthored publications
for the period `2007-2023`. Round the percentage values to two (`2`) decimal places. Assign the
values to the variables `umsi_coauth_2007_2023_pct` and `non_umsi_coauth_2007_2023_pct`,
respectively.

In [22]:
umsi_coauth_2007_2023_pct = round(
    len(umsi_coauth_pubs_2007_2023) / len(pubs_2007_2023) * 100, 2
)

non_umsi_coauth_2007_2023_pct = round(
    len(non_umsi_coauth_pubs_2007_2023) / len(pubs_2007_2023) * 100, 2
)

print(f"UMSI coauthored pubs: {umsi_coauth_2007_2023_pct}%")
print(f"Non-UMSI coauthored pubs: {non_umsi_coauth_2007_2023_pct}%")

UMSI coauthored pubs: 48.57%
Non-UMSI coauthored pubs: 51.43%


In [23]:
# Hidden tests are within this cell

### 8.3 Citation counts, 2007-2023 [1 pt]

Sum the citation counts for all publications, UMSI coauthored publications, and non-UMSI coauthored publications for the period `2007` and `2023` (inclusive). Assign the sums computed to the following variables:

* `total_cits_2007_2023`
* `total_umsi_coauth_cits_2007_2023`
* `total_non_umsi_coauth_cits_2007_2023`

__Hint__: Iterate over a slice of `years`, call `lab.count_citations_per_annum()` during each iteration of the loop, accumulate a list of values, and then sum the list.

In [24]:
years_2007_2023 = [yr for yr in years if 2007 <= yr <= 2023]

total_cits_2007_2023 = sum(
    lab.count_citations_per_annum(pubs_2007_2023, yr)
    for yr in years_2007_2023
)

total_umsi_coauth_cits_2007_2023 = sum(
    lab.count_citations_per_annum(umsi_coauth_pubs_2007_2023, yr)
    for yr in years_2007_2023
)

total_non_umsi_coauth_cits_2007_2023 = sum(
    lab.count_citations_per_annum(non_umsi_coauth_pubs_2007_2023, yr)
    for yr in years_2007_2023
)

print(f"Total citations 2007-2023: {total_cits_2007_2023}")
print(f"UMSI coauthor citations:   {total_umsi_coauth_cits_2007_2023}")
print(f"Non-UMSI citations:        {total_non_umsi_coauth_cits_2007_2023}")

Total citations 2007-2023: 1247
UMSI coauthor citations:   663
Non-UMSI citations:        584


In [25]:
# Hidden tests are within this cell

### 8.4 Group citation count percentages [1 pt]

Compute the percentage values of UMSI coauthored publications and non-UMSI coauthored publications
for the period `2007-2023`. Round the percentage values to two (`2`) decimal places. Assign the
values to the variables `total_umsi_coauth_cits_2007_2023_pct` and `total_non_umsi_coauth_cits_2007_2023_pct`, respectively.

In [26]:
total_umsi_coauth_cits_2007_2023_pct = round(
    total_umsi_coauth_cits_2007_2023 / total_cits_2007_2023 * 100, 2
)

total_non_umsi_coauth_cits_2007_2023_pct = round(
    total_non_umsi_coauth_cits_2007_2023 / total_cits_2007_2023 * 100, 2
)

print(f"UMSI coauthor citation share:     {total_umsi_coauth_cits_2007_2023_pct}%")
print(f"Non-UMSI coauthor citation share: {total_non_umsi_coauth_cits_2007_2023_pct}%")

UMSI coauthor citation share:     53.17%
Non-UMSI coauthor citation share: 46.83%


In [27]:
# Hidden tests are within this cell

## 9.0 Create the chart data (2007-2023) [1 pt]

Adopt the same approach used to generate the chart data for the previous stacked bar chart. Utilize
the following lists:

* `pubs_2007_2023`
* `umsi_coauth_pubs_2007_2023`
* `non_umsi_coauth_pubs_2007_2023`

Call the function `lab.create_chart_data()` three times and pass the arguments required to return three
new lists representing citation counts for all publications, UMSI coauthored publications, and
non-UMSI coauthored publications published between `2007` and `2023`. Assign the return values to
the following variables:

* `chrt_all_cits_2007_2023`
* `chrt_umsi_coauth_cits_2007_2023`
* `chrt_non_umsi_coauth_cits_2007_2023`

In [28]:
chrt_all_cits_2007_2023 = lab.create_chart_data(pubs_2007_2023, years)
chrt_umsi_coauth_cits_2007_2023 = lab.create_chart_data(umsi_coauth_pubs_2007_2023, years, "UMSI coauthors")
chrt_non_umsi_coauth_cits_2007_2023 = lab.create_chart_data(non_umsi_coauth_pubs_2007_2023, years, "No UMSI coauthors")

In [29]:
# Hidden tests are within this cell

## 10.0 Generate a second stacked bar chart

Run the notebook to generate the stacked bar chart. If the chart is not displayed, switch to OILER
mode and debug your code.

In [30]:
# Generate stacked bar chart
grouped_cits = chrt_umsi_coauth_cits_2007_2023 + chrt_non_umsi_coauth_cits_2007_2023
title = "Citation counts: Paul Resnick authored/coauthored publications, 2007-2023"
subtitle = (
    f"publications: n={len(pubs_2007_2023)} | "
    f"citations: n={total_cits_2007_2023} | three-year rolling mean"
)
custom_colors = ["#FFCB05", "#00274C"]  # maize, blue
custom_labels = {
    "UMSI coauthors": f"UMSI coauthors (n={len(umsi_coauth_pubs_2007_2023)})",
    "No UMSI coauthors": f"No UMSI coauthors (n={len(non_umsi_coauth_pubs_2007_2023)})",
}

chart = bar.create_stacked_bar_chart(
    data_grouped=grouped_cits,
    data_ungrouped=chrt_all_cits_2007_2023,
    x_shorthand="year:O",
    y_shorthand="citations:Q",
    grp_shorthand="group:N",
    custom_colors=custom_colors,
    custom_labels=custom_labels,
    title=title,
    subtitle=subtitle,
    rm_shorthand="rolling_mean:Q",
    width=600,
)
chart.display()

alt.LayerChart(...)

## 11.0 Write to file [1 pt]

End this analysis by writing the list of UMSI coauthored publications to a JSON file in the `data` directory called `stu-resnick-citations-umsi_coauthors.json`. Compare the file you produce against the test fixture file `fxt-resnick-citations-umsi_coauthors.json`. If the files are identical, declare victory. If not switch back to OILER mode.

In [31]:
output_filepath = "./data/stu-resnick-citations-umsi_coauthors.json"
with open(output_filepath, "w", encoding="utf-8") as file_obj:
    json.dump(umsi_coauth_pubs, file_obj, indent=2)

print(f"File written: {output_filepath}")

File written: ./data/stu-resnick-citations-umsi_coauthors.json


In [32]:
# Hidden tests are within this cell